# Brain Tumor MRI Classification — ViT-B/16 (pretrained, fine-tuned)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Why this version is different from the earlier ViT run: in the old notebook, `transformers` printed
`MISSING` for **every** backbone weight (`vit.layers.{0...11}...`), so the "pretrained" ViT actually
started from random weights — that's why it only reached ~57%. This notebook uses **timm**, which loads the
pretrained ViT-B/16 weights reliably, and fine-tunes the whole network with a warmup + cosine schedule
and mixed precision.

At the end it exports the model to ONNX for the Streamlit app and downloads `vit_b16.onnx`.

In [ ]:
!pip install -q timm kagglehub onnx onnxruntime

import os, copy, time, math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
assert device.type == "cuda", "No GPU! Go to Runtime > Change runtime type > T4 GPU, then run again."

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

In [ ]:
# ---------------- Dataset ----------------
import kagglehub
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
train_path = os.path.join(path, "Training")
test_path = os.path.join(path, "Testing")
print(os.listdir(train_path))

In [ ]:
# ---------------- Model config ----------------
MODEL_NAME = "vit_base_patch16_224.augreg2_in21k_ft_in1k"
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4

cfg = timm.data.resolve_data_config({}, model=timm.create_model(MODEL_NAME, pretrained=False))
mean, std = list(cfg["mean"]), list(cfg["std"])
print("mean:", mean, "std:", std)   # should be 0.5 / 0.5 (the Streamlit app assumes this)

In [ ]:
# ---------------- Transforms + datasets ----------------
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Two views of the same folder so validation never gets augmentation
train_full = datasets.ImageFolder(train_path, transform=train_transform)
val_full = datasets.ImageFolder(train_path, transform=eval_transform)
class_labels = train_full.classes
print(train_full.class_to_idx)   # {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}

# Same 80/20 split as the other models
idx = np.random.RandomState(SEED).permutation(len(train_full))
n_val = int(0.2 * len(idx))
val_idx, train_idx = idx[:n_val], idx[n_val:]

train_ds = Subset(train_full, train_idx)
val_ds = Subset(val_full, val_idx)
test_ds = datasets.ImageFolder(test_path, transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

In [ ]:
# ---------------- Build model (pretrained weights actually load here) ----------------
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)
print("Parameters: %.1fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

In [ ]:
# ---------------- Training setup ----------------
EPOCHS = 12
WARMUP_EPOCHS = 1
PATIENCE = 4

head_params = list(model.get_classifier().parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

# Lower LR for pretrained backbone, higher LR for the new classifier head
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 3e-5},
    {"params": head_params, "lr": 3e-4},
], weight_decay=0.05)

steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler("cuda")


def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total

In [ ]:
# ---------------- Train ----------------
history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
best_acc, best_state, no_improve = 0.0, None, 0

for epoch in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader)
    for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
        history[k].append(v)
    print(f"Epoch {epoch+1}/{EPOCHS} - loss {tr_loss:.4f} - acc {tr_acc:.4f} "
          f"- val_loss {va_loss:.4f} - val_acc {va_acc:.4f} - {time.time()-t0:.0f}s")

    if va_acc > best_acc:
        best_acc, best_state, no_improve = va_acc, copy.deepcopy(model.state_dict()), 0
        print("  ✓ best so far")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(best_state)
print(f"Best val accuracy: {best_acc:.4f}")

In [ ]:
# ---------------- Training curves ----------------
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history["accuracy"], label="Train Acc"); plt.plot(history["val_accuracy"], label="Val Acc")
plt.xlabel("Epoch"); plt.legend(); plt.title("ViT-B/16 Accuracy")
plt.subplot(1, 2, 2)
plt.plot(history["loss"], label="Train Loss"); plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch"); plt.legend(); plt.title("ViT-B/16 Loss")
plt.show()

In [ ]:
# ---------------- Evaluate on the held-out test set ----------------
test_loss, test_acc = run_epoch(test_loader)
print(f"Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.4f}")

model.eval()
y_true, y_pred = [], []
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    for images, labels in test_loader:
        y_pred.extend(model(images.to(device)).argmax(1).cpu().numpy())
        y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_labels, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - ViT-B/16")
plt.show()

In [ ]:
# ---------------- Save PyTorch weights (backup) ----------------
torch.save(model.state_dict(), "vit_b16_brain_tumor.pth")

# Optional but recommended: also keep a copy in Google Drive so it survives a Colab reset
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/brain_tumor_models", exist_ok=True)
    torch.save(model.state_dict(), "/content/drive/MyDrive/brain_tumor_models/vit_b16_brain_tumor.pth")
    print("Saved a copy to Google Drive: MyDrive/brain_tumor_models/")
except Exception as e:
    print("Drive backup skipped:", e)

In [ ]:
# ---------------- Export to ONNX for the Streamlit app ----------------
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

export_model = copy.deepcopy(model).float().cpu().eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
export_kwargs = dict(input_names=["pixel_values"], output_names=["logits"],
                     dynamic_axes={"pixel_values": {0: "batch"}, "logits": {0: "batch"}},
                     opset_version=17)
try:
    torch.onnx.export(export_model, (dummy,), "vit_b16_fp32.onnx", dynamo=False, **export_kwargs)
except TypeError:   # older torch without the dynamo argument
    torch.onnx.export(export_model, (dummy,), "vit_b16_fp32.onnx", **export_kwargs)
onnx.checker.check_model("vit_b16_fp32.onnx")

quantize_dynamic("vit_b16_fp32.onnx", "vit_b16_int8.onnx", weight_type=QuantType.QInt8)
for f in ["vit_b16_fp32.onnx", "vit_b16_int8.onnx"]:
    print(f, "%.0f MB" % (os.path.getsize(f) / 1e6))

In [ ]:
# ---------------- Check the ONNX models are as accurate as PyTorch ----------------
# (400 test images, 100 per class - running all 1600 on Colab's CPU would be slow)
sub_idx = [i for c in range(NUM_CLASSES) for i in np.where(np.array(test_ds.targets) == c)[0][:100]]
sub_loader = DataLoader(Subset(test_ds, sub_idx), batch_size=16, shuffle=False)

def onnx_accuracy(path):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    correct = total = 0
    for images, labels in sub_loader:
        logits = sess.run(None, {"pixel_values": images.numpy()})[0]
        correct += (logits.argmax(1) == labels.numpy()).sum(); total += len(labels)
    return correct / total

with torch.no_grad():
    pt_correct = sum((export_model(x).argmax(1) == y).sum().item() for x, y in sub_loader)
pt_acc = pt_correct / len(sub_idx)
fp32_acc = onnx_accuracy("vit_b16_fp32.onnx")
int8_acc = onnx_accuracy("vit_b16_int8.onnx")
print(f"PyTorch {pt_acc:.4f} | ONNX fp32 {fp32_acc:.4f} | ONNX int8 {int8_acc:.4f}")

# Use the small int8 file unless quantization cost more than 1% accuracy
chosen = "vit_b16_int8.onnx" if int8_acc >= fp32_acc - 0.01 else "vit_b16_fp32.onnx"
import shutil
shutil.copy(chosen, "vit_b16.onnx")
print("Using", chosen, "-> saved as vit_b16.onnx (%.0f MB)" % (os.path.getsize("vit_b16.onnx") / 1e6))

In [ ]:
# ---------------- Download the file for the Streamlit app ----------------
from google.colab import files
files.download("vit_b16.onnx")